# Dự báo nhiệt độ với T2f
Mở từ repo đã giải nén hoặc thêm `t2f-forecasting.zip` vào Kaggle/Colab.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

work = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
roots = [Path.cwd(), Path.cwd().parent, Path("/kaggle/input")]
repo = next((p for p in roots if (p / "run_experiment.py").is_file()), None)
if repo is None:
    repo = next((p.parent for p in Path("/kaggle/input").rglob("run_experiment.py") if (p.parent / "src/experiment.py").is_file()), None)
if repo is None:
    search = [work, Path("/kaggle/input")]
    archive = next((p for base in search for p in base.rglob("t2f-forecasting.zip")), None)
    if archive is None and "google.colab" in sys.modules:
        from google.colab import files
        archive = Path(next(iter(files.upload())))
    if archive is None:
        raise FileNotFoundError("Hãy thêm t2f-forecasting.zip vào notebook.")
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(work)
    repo = work / "t2f-forecasting"
print(repo)

In [ ]:
datasets = ["ETTh", "ETTm"]
modes = ["level", "delta", "delta_ridge"]
seeds = [42, 43, 44]
profile = "main"

output = work / "results" / "runs"
command = [sys.executable, str(repo / "run_experiment.py"), "--datasets", *datasets,
           "--modes", *modes, "--seeds", *map(str,seeds), "--profile", profile,
           "--output", str(output)]
subprocess.run(command, check=True, env={**os.environ, "OPENBLAS_NUM_THREADS": "1"})
print(shutil.make_archive(str(work / f"t2f_results_{profile}"), "zip", output / profile))